## 5-Task MNIST Class-IL Example - Multi-Seed Evaluation

In [3]:
import torch
import torch.optim as optim
import numpy as np

from networks.BP_network import BP_network
from networks.EWC_network import EWC_network
from networks.EFC_network import EFC_network
# from src.dataloaders import ClassILMNIST5Task
from src.dataloaders_2 import ClassILMNIST5Task, ClassILCIFAR5Task, TaskILMNIST
from src.utils import dotdict

from tqdm import tqdm
from collections import defaultdict

# ============================================================================
# Configuration
# ============================================================================
N_SEEDS = 5  # Number of random seeds to run
SEEDS = list(range(N_SEEDS))  # Seeds: 0, 1, 2, ..., N_SEEDS-1

base_config = dotdict({
    "setting": "ClassILMNIST5Task",
    "num_tasks": 5,
    "classes_per_task": 2,
    "batch_size": 256,
    "epochs": 5,
    "loss_fn": "ce",
    "scheduler": "CosineAnnealingLR",
    "output_dir": "./outputs",
    "seed": 0,  # Will be overwritten per run
    "optimizer": "Adam",
    "num_workers": 0,
    "mode": "di",
    "lr": 1e-5,
    "target_lr": 1e-1,
    "alpha_di": 0.0017,
    "alpha_I": 0.0017,
    "tau": 0.032,
    "dt_di": 0.02,
    "psi_lr": 0.1,
    "alpha_psi": 0.0,
    "time_constant_ratio": 0.2,
    "tmax_di": 500,
    "flatten_imgs": True,
    "k_p": 2.0,
    "eps": 1e-4,
    "save": False,
    "importance_ewc": 4.0,
    "beta_efc": 100.0,
    "layers": [784, 100, 100, 10],
    # "layers": [512, 100, 100, 10],
    "device": "cuda" if torch.cuda.is_available() else "cpu",
})


def configure_cnn_encoder(config):
    """Automatically configure CNN encoder for CIFAR and TinyImageNet datasets."""
    setting = config.get("setting", "")
    if "CIFAR" in setting or "TinyImageNet" in setting:
        config.use_cnn_encoder = True
        config.cnn_encoder = "resnet18"
        config.cnn_pretrained = True
        config.encoder_freeze = True
    return config


base_config = configure_cnn_encoder(base_config)
print(base_config)

{'setting': 'ClassILMNIST5Task', 'num_tasks': 5, 'classes_per_task': 2, 'batch_size': 256, 'epochs': 5, 'loss_fn': 'ce', 'scheduler': 'CosineAnnealingLR', 'output_dir': './outputs', 'seed': 0, 'optimizer': 'Adam', 'num_workers': 0, 'mode': 'di', 'lr': 1e-05, 'target_lr': 0.1, 'alpha_di': 0.0017, 'alpha_I': 0.0017, 'tau': 0.032, 'dt_di': 0.02, 'psi_lr': 0.1, 'alpha_psi': 0.0, 'time_constant_ratio': 0.2, 'tmax_di': 500, 'flatten_imgs': True, 'k_p': 2.0, 'eps': 0.0001, 'save': False, 'importance_ewc': 4.0, 'beta_efc': 100.0, 'layers': [784, 100, 100, 10], 'device': 'cuda'}


In [4]:
# ============================================================================
# Helper Functions
# ============================================================================
def set_seed(seed):
    """Set random seed for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def evaluate(network, test_loader, task_id, task_classes):
    """Evaluate network on test set for specified classes."""
    network.eval()
    correct = 0
    total = 0
    class_start, class_end = task_classes
    
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(network.device), y.to(network.device)
            labels = y.argmax(dim=1)
            
            mask = (labels >= class_start) & (labels <= class_end)
            if mask.sum() == 0:
                continue
            
            x_masked = x[mask]
            labels_masked = labels[mask]
            
            y_hat = network(x_masked)
            preds = y_hat[:, class_start:class_end+1].argmax(dim=1) + class_start
            
            correct += (preds == labels_masked).sum().item()
            total += mask.sum().item()
    
    accuracy = correct / total if total > 0 else 0.0
    return accuracy, total


def least_square_initialization(network, dataloader, task_id, classes_per_task=2, weight_decay=1e-4):
    """Least-square optimal initialization for new classifier weights."""
    network.eval()
    new_start = task_id * classes_per_task
    new_end = (task_id + 1) * classes_per_task
    
    features_list = []
    labels_list = []
    
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(network.device)
            features = x
            for layer in network.layers[:-1]:
                features = layer(features)
            features_list.append(features)
            labels_list.append(y.argmax(dim=1))
    
    features = torch.cat(features_list, dim=0)
    labels = torch.cat(labels_list, dim=0)
    
    N, d = features.shape
    features_ext = torch.cat([features, torch.ones(N, 1, device=features.device)], dim=1)
    
    num_new_classes = new_end - new_start
    targets = torch.zeros(N, num_new_classes, device=features.device)
    for i, label in enumerate(labels):
        if new_start <= label < new_end:
            targets[i, label - new_start] = 1.0
    
    mask = (labels >= new_start) & (labels < new_end)
    features_new = features_ext[mask]
    targets_new = targets[mask]
    
    ZtZ = features_new.T @ features_new
    ZtY = features_new.T @ targets_new
    
    reg = weight_decay * features_new.shape[0] * torch.eye(d + 1, device=features.device)
    W_ls = torch.linalg.solve(ZtZ + reg, ZtY)
    
    with torch.no_grad():
        for c_idx, c in enumerate(range(new_start, new_end)):
            network.layers[-1]._weights[c] = W_ls[:d, c_idx]
            network.layers[-1]._bias[c] = W_ls[d, c_idx]


def train_continual(network_class, config, train_loaders, full_test_loader, name, verbose=True):
    """Train a network on all 5 tasks sequentially with fixed epochs per task."""
    if verbose:
        print(f"\n{'='*70}")
        print(f"Training: {name}")
        print(f"{'='*70}")
    
    net = network_class(config).to(config.device)
    
    results = {
        'name': name,
        'task_accuracies': [],
        'training_history': [],
    }
    
    for task_id in range(config.num_tasks):
        if verbose:
            print(f"\n--- Task {task_id} (classes {task_id*2}-{task_id*2+1}) ---")
        
        net.task_id = task_id
        seen_classes_end = (task_id + 1) * config.classes_per_task - 1
        
        if task_id > 0 and "classil" in config.setting.lower():
            least_square_initialization(net, train_loaders[task_id], task_id, config.classes_per_task)
            if hasattr(net, '_first_task'):
                net._first_task = False
        
        optimizer = optim.Adam(net.parameters(), lr=config.lr)
        if verbose:
            print(f"  Training for {config.epochs} epochs")
        
        for epoch in range(config.epochs):
            net.train()
            pbar = tqdm(total=len(train_loaders[task_id]), 
                        desc=f"  Epoch {epoch+1}", unit="batch", leave=False, disable=not verbose)
            
            for x, y in train_loaders[task_id]:
                x, y = x.to(config.device), y.to(config.device)
                optimizer.zero_grad()
                y_hat = net(x)
                _ = net.calculate_loss(y_hat, y.argmax(dim=1))
                net.backward(y)
                optimizer.step()
                pbar.update(1)
            pbar.close()
            
            net.eval()
            combined_acc, _ = evaluate(net, full_test_loader, task_id=task_id, 
                                       task_classes=[0, seen_classes_end])
            
            task_accs = {}
            for t in range(task_id + 1):
                t_start = t * config.classes_per_task
                t_end = t_start + config.classes_per_task - 1
                acc, _ = evaluate(net, full_test_loader, task_id=t, task_classes=[t_start, t_end])
                task_accs[f'task_{t}'] = acc
            
            results['training_history'].append({
                'task_id': task_id,
                'epoch': epoch + 1,
                'combined_acc': combined_acc,
                **task_accs
            })
            
            if verbose:
                acc_str = " | ".join([f"T{t}={task_accs[f'task_{t}']:.3f}" for t in range(task_id + 1)])
                print(f"  Epoch {epoch+1:2d}: Combined={combined_acc:.4f} | {acc_str}")
        
        net.complete_task(train_loaders[task_id])
        
        final_accs = {'after_task': task_id, 'combined': combined_acc}
        for t in range(task_id + 1):
            t_start = t * config.classes_per_task
            t_end = t_start + config.classes_per_task - 1
            acc, _ = evaluate(net, full_test_loader, task_id=t, task_classes=[t_start, t_end])
            final_accs[f'task_{t}'] = acc
        results['task_accuracies'].append(final_accs)
        
        if verbose:
            print(f"  >> Task {task_id} complete. Combined acc: {combined_acc:.4f}")
    
    return results

In [ ]:
# ============================================================================
# Run Experiments Over Multiple Seeds
# ============================================================================
torch.set_default_device(base_config.device)

# Methods to evaluate
methods = {
    'BP': BP_network,
    # 'EWC': EWC_network,
    # 'EFC': EFC_network,
}

# Store results: {method_name: {seed: results_dict}}
all_seed_results = {name: {} for name in methods.keys()}

for seed in SEEDS:
    print(f"\n{'#'*70}")
    print(f"# SEED {seed}")
    print(f"{'#'*70}")
    
    # Set seed and create config for this run
    set_seed(seed)
    config = dotdict({**base_config, 'seed': seed})
    
    # Create dataloaders (with this seed)
    dataloader = ClassILMNIST5Task(config)
    # dataloader = ClassILCIFAR5Task(config)
    # dataloader = TaskILMNIST(config)
    train_loaders = []
    test_loaders = []
    for task_id in range(config.num_tasks):
        train_loader, test_loader = dataloader.get_dataloaders(task_id=task_id)
        train_loaders.append(train_loader)
        test_loaders.append(test_loader)
    full_test_loader = test_loaders[-1]
    
    # Train each method
    for method_name, network_class in methods.items():
        set_seed(seed)  # Reset seed before each method for fair comparison
        results = train_continual(network_class, config, train_loaders, full_test_loader, 
                                  f"{method_name} (seed={seed})", verbose=True)
        all_seed_results[method_name][seed] = results


######################################################################
# SEED 0
######################################################################
DataLoader using device: cuda

Training: EFC (seed=0)

--- Task 0 (classes 0-1) ---
  Training for 5 epochs


  Epoch  1: Combined=0.5038 | T0=0.504


  Epoch  2: Combined=0.4760 | T0=0.476


  Epoch  3: Combined=0.4291 | T0=0.429


  Epoch  4: Combined=0.3843 | T0=0.384


...

  Epoch  3: Combined=0.7352 | T0=0.735 | T1=0.000 | T2=0.000 | T3=0.000 | T4=0.000


  Epoch  4: Combined=0.7902 | T0=0.790 | T1=0.000 | T2=0.000 | T3=0.000 | T4=0.000


  Epoch  5: Combined=0.8285 | T0=0.829 | T1=0.000 | T2=0.000 | T3=0.000 | T4=0.000


Fisher: 100%|██████████| 47/47 [00:00<00:00, 184.88it/s]


  >> Task 4 complete. Combined acc: 0.8285


In [6]:
# ============================================================================
# Results Per Seed
# ============================================================================
print("\n" + "="*70)
print("RESULTS PER SEED")
print("="*70)

for method_name in methods.keys():
    print(f"\n{method_name}:")
    print("-" * 70)
    header = f"{'Seed':<6} | " + " | ".join([f"T{t}" for t in range(base_config.num_tasks)]) + " | Combined"
    print(header)
    print("-" * 70)
    
    for seed in SEEDS:
        final = all_seed_results[method_name][seed]['task_accuracies'][-1]
        task_accs = " | ".join([f"{final[f'task_{t}']:.3f}" for t in range(base_config.num_tasks)])
        print(f"{seed:<6} | {task_accs} | {final['combined']:.4f}")


RESULTS PER SEED

EFC:
----------------------------------------------------------------------
Seed   | T0 | T1 | T2 | T3 | T4 | Combined
----------------------------------------------------------------------
0      | 0.885 | 0.000 | 0.000 | 0.000 | 0.000 | 0.8850
1      | 0.820 | 0.000 | 0.000 | 0.000 | 0.000 | 0.8200
2      | 0.801 | 0.000 | 0.000 | 0.000 | 0.000 | 0.8013
3      | 0.790 | 0.000 | 0.000 | 0.000 | 0.000 | 0.7897
4      | 0.829 | 0.000 | 0.000 | 0.000 | 0.000 | 0.8285


In [7]:
# ============================================================================
# Aggregated Results (Mean ± Std)
# ============================================================================
print("\n" + "="*70)
print("AGGREGATED RESULTS (Mean ± Std over seeds)")
print("="*70)

aggregated_results = {}

for method_name in methods.keys():
    # Collect final accuracies across seeds
    combined_accs = []
    task_accs = {t: [] for t in range(base_config.num_tasks)}
    
    for seed in SEEDS:
        final = all_seed_results[method_name][seed]['task_accuracies'][-1]
        combined_accs.append(final['combined'])
        for t in range(base_config.num_tasks):
            task_accs[t].append(final[f'task_{t}'])
    
    aggregated_results[method_name] = {
        'combined_mean': np.mean(combined_accs),
        'combined_std': np.std(combined_accs),
        'task_means': {t: np.mean(task_accs[t]) for t in range(base_config.num_tasks)},
        'task_stds': {t: np.std(task_accs[t]) for t in range(base_config.num_tasks)},
    }

# Print aggregated results
print("\nFinal combined accuracy (all 10 classes) after Task 4:")
print("-" * 50)
for method_name, agg in aggregated_results.items():
    print(f"{method_name:20s}: {agg['combined_mean']:.4f} ± {agg['combined_std']:.4f}")

print("\nPer-task accuracy breakdown after all tasks:")
print("-" * 90)
header = f"{'Method':<12} | " + " | ".join([f"{'T'+str(t):^13}" for t in range(base_config.num_tasks)]) + " | Combined"
print(header)
print("-" * 90)

for method_name, agg in aggregated_results.items():
    task_strs = []
    for t in range(base_config.num_tasks):
        task_strs.append(f"{agg['task_means'][t]:.3f}±{agg['task_stds'][t]:.3f}")
    task_line = " | ".join(task_strs)
    print(f"{method_name:<12} | {task_line} | {agg['combined_mean']:.3f}±{agg['combined_std']:.3f}")


AGGREGATED RESULTS (Mean ± Std over seeds)

Final combined accuracy (all 10 classes) after Task 4:
--------------------------------------------------
EFC                 : 0.8249 ± 0.0330

Per-task accuracy breakdown after all tasks:
------------------------------------------------------------------------------------------
Method       |      T0       |      T1       |      T2       |      T3       |      T4       | Combined
------------------------------------------------------------------------------------------
EFC          | 0.825±0.033 | 0.000±0.000 | 0.000±0.000 | 0.000±0.000 | 0.000±0.000 | 0.825±0.033


In [7]:
# ============================================================================
# Forgetting Analysis (Mean ± Std)
# ============================================================================
print("\n" + "="*70)
print("FORGETTING ANALYSIS (Mean ± Std over seeds)")
print("="*70)
print("Forgetting = max accuracy on task during training - final accuracy on task")
print("-" * 70)

for method_name in methods.keys():
    print(f"\n{method_name}:")
    
    forgetting_per_task = {t: [] for t in range(base_config.num_tasks - 1)}  # No forgetting for last task
    
    for seed in SEEDS:
        results = all_seed_results[method_name][seed]
        final_accs = results['task_accuracies'][-1]
        
        # For each task (except the last), find max accuracy achieved during its training
        for t in range(base_config.num_tasks - 1):
            # Get accuracies for task t across all epochs when it was being trained
            max_acc = 0.0
            for hist in results['training_history']:
                if f'task_{t}' in hist:
                    max_acc = max(max_acc, hist[f'task_{t}'])
            
            forgetting = max_acc - final_accs[f'task_{t}']
            forgetting_per_task[t].append(forgetting)
    
    # Print forgetting stats
    header = "  " + " | ".join([f"T{t}" for t in range(base_config.num_tasks - 1)]) + " | Avg"
    print(header)
    
    means = [np.mean(forgetting_per_task[t]) for t in range(base_config.num_tasks - 1)]
    stds = [np.std(forgetting_per_task[t]) for t in range(base_config.num_tasks - 1)]
    avg_forgetting = np.mean(means)
    
    forgetting_strs = [f"{means[t]:.3f}±{stds[t]:.3f}" for t in range(base_config.num_tasks - 1)]
    print("  " + " | ".join(forgetting_strs) + f" | {avg_forgetting:.3f}")


FORGETTING ANALYSIS (Mean ± Std over seeds)
Forgetting = max accuracy on task during training - final accuracy on task
----------------------------------------------------------------------

EFC:
  T0 | T1 | T2 | T3 | Avg
  0.000±0.000 | 0.001±0.001 | 0.000±0.000 | 0.000±0.000 | 0.000
